In [ ]:
from ppg_basis import ppgGenerator
from ppg_basis import ppgExtractor
import matplotlib.pyplot as plt
from numba import njit
import numpy as np

In [ ]:
# Generate signal
ppgGen = ppgGenerator(fs=125,
                      hr=60,
                      mu=1,
                      sigma=0,
                      duration=10,
                      L=3,
                      basis_type="gamma",
                      ode_solver="rk3")

sig = ppgGen.generate_signal()
plt.plot(sig)
plt.xlabel("Time")
plt.ylabel("Intensity")
plt.title("Original PPG")

In [ ]:
# Define custom cost function
@njit
def custom_cost(model, signal):
    cost = np.sum((model - signal)**2)
    return cost

In [ ]:
# Extract signal parameters
# pass custom cost function as parameter in ppgExtractor
ppgExt = ppgExtractor(signal=sig,
                      fs=125,
                      hr=60,
                      sigma=0,
                      L=3,
                      basis_type='gamma',
                      ode_solver="rk3",
                      cost_func=custom_cost)

# Select additional cost metrics you want to examine
views = ppgExt.plot_cost_landscape()

In [ ]:
for v in views:
    display(v)

In [ ]:
ppgExt.cost_metrics = ["mse", "corr"]
theta_pred, params_pred = ppgExt.extract_ppg(block_update=True, 
                                             coord_cycles=4)

In [ ]:
# Generate PPG using extracted parameters
ppgPrd = ppgGenerator(fs=125,
                      hr=60,
                      mu=1,
                      sigma=0,
                      duration=10,
                      L=3,
                      basis_type="gamma",
                      thetas=theta_pred,
                      params=params_pred,
                      ode_solver="rk3")
pred = ppgPrd.generate_signal()

plt.plot(sig)
plt.plot(pred)
plt.xlabel("Time")
plt.ylabel("Intensity")
plt.title("Original vs Predicted PPG")
plt.legend(["Original", "Predicted"])